# Acupuncture utilization: age buckets and rurality

Fit a patient-level ridge logistic model using the requested age buckets:
<35, 35-44, 45-54, 55-64, 65-74 and 75+.
Age comparisons use <35 as the reference; RUCC comparisons use code 6.

This notebook is self-contained. It replaces the age spline with categorical age
and includes gender, recorded race/ethnicity, rurality and condition indicators.
The original spline notebook is preserved.

Start with the synthetic demonstration. For your data, set USE_SYNTHETIC_DEMO
to False and edit INPUT_CSV. Restart the kernel and run all cells to obtain new
coefficients, validation results and direct adjusted odds-ratio tables.
Synthetic outputs are examples, not results for your patient population.

## 1. Dependencies

Use Python 3.10 or later. If needed, uncomment and run the installation line below,
then restart the kernel. All model/data processing occurs locally.

In [ ]:
# Uncomment only if these packages are missing in your notebook environment:
# %pip install "numpy>=1.24,<3" "pandas>=2,<4" "scikit-learn>=1.3,<2" "joblib>=1.3,<2" matplotlib

In [3]:
from __future__ import annotations
import json
import warnings
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import sklearn

from sklearn.base import clone
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    log_loss,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    FunctionTransformer,
    OneHotEncoder,
    StandardScaler,
)
import matplotlib.pyplot as plt
from IPython.display import display

import pyodbc
import urllib
from sqlalchemy import create_engine

warnings.filterwarnings('error', category=ConvergenceWarning)
print('scikit-learn:', sklearn.__version__)

scikit-learn: 1.9.1


## IMport the data

In [4]:
%%time
# establish the connection parameters
odbc_str = ("DRIVER={ODBC Driver 18 for SQL Server};"
            "SERVER=VHACDWA01.vha.med.va.gov,1433;"
            "DATABASE=OPCCCT_CIH;"
            "Trusted_Connection=yes;"
            "Encrypt=yes;"
            "TrustServerCertificate=no;")

# connect
params = urllib.parse.quote_plus(odbc_str)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

# read in the data
df = pd.read_sql("SELECT * FROM [OPCCCT_CIH].[dflt].[logit_patient_data]", engine)
df = df.loc[df['gender'].notnull()]

CPU times: total: 3min 33s
Wall time: 7min 53s


## 2. Configuration

The six age buckets have fixed cutoffs. Age 35 enters 35-44, age 45 enters 45-54,
and age 75 enters 75+. Missing age and rurality remain explicit Unknown categories.

The default is condition presence only, matching the latest model summary.
Set USE_COUNTS to True only if condition-specific diagnosis counts are intended
predictors. Their clinical time-window requirements remain the same.

 recognizes your PatientICN, Condition, diagnosis_count and race_mod
names, with fallbacks for the original notebook's canonical names. Edit mappings
if needed. Rurality must contain RUCC 1-9 or missing/Unknown values.

An optional full roster keeps people with no listed conditions. Missing addresses
do not cause this notebook to drop a person; however, patients removed before
export must be supplied through the input or roster to be included.

In [26]:
USE_SYNTHETIC_DEMO = False

AGE_LABELS = ["<35", "35-44", "45-54", "55-64", "65-74", "75+"]
AGE_REFERENCE = "<35"
RURALITY_REFERENCE = "6"
GENDER_REFERENCE = "Male"
RACE_REFERENCE = "WHITE"

USE_COUNTS = False
DUPLICATE_COUNTS = "sum"
OUTER_FOLDS = 5
RANDOM_SEED = 42
JOBS = 1
TARGET_CSV = None
TARGET_ROSTER_CSV = None

OUTPUT_BASE = Path.cwd() / "acupuncture_age_bucket_outputs"
run_name = "synthetic_demo" if USE_SYNTHETIC_DEMO else "patient_analysis"
run_name += "_counts" if USE_COUNTS else "_presence"
OUTPUT_DIR = OUTPUT_BASE / run_name
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Mode:", "SYNTHETIC DEMO DATA" if USE_SYNTHETIC_DEMO else "YOUR INPUT DATA")
print("Age reference:", AGE_REFERENCE, "; RUCC reference:", RURALITY_REFERENCE)
print("Output folder:", OUTPUT_DIR.resolve())

Mode: YOUR INPUT DATA
Age reference: <35 ; RUCC reference: 6
Output folder: C:\Users\VACOGRAUPE\OneDrive - Department of Veterans Affairs\Desktop\acupuncture_python_code\acupuncture_age_bucket_outputs\patient_analysis_presence


## 3. Input checks and long-to-wide transformation

Required predictors are patient ID, age, gender, race/race_mod and rurality/RUCC.
The long table also needs condition/Condition. The training outcome is
current_acupuncture_user; optional scoring data may omit the outcome.
times_diagnosed/diagnosis_count may be missing for presence-only modeling.

Age buckets are created from raw age in completed years. Unknown values remain
Unknown. Patients are checked for consistent demographics and outcomes across
their condition records. Each modeled condition becomes a presence indicator.
No recorded diagnosis is its reference; that does not establish clinical absence.

In [24]:
%%time
ID = "PatientICN"
OUTCOME = "current_acupuncture_user"
PERSON_FIELDS = ["age", "gender", "race_mod", "rurality", OUTCOME]

def add_age_buckets(wide: pd.DataFrame) -> pd.DataFrame:
    values = pd.to_numeric(wide["age"], errors="raise").astype(float)
    if (not np.isfinite(values.dropna()).all()) or values.dropna().lt(0).any():
        raise ValueError("Age must be nonnegative and finite, or missing.")
    wide["age_bucket"] = pd.cut(
        values, bins=[-np.inf, 35, 45, 55, 65, 75, np.inf],
        labels=AGE_LABELS, right=False,
    ).astype(object).fillna("Unknown")
    wide["rurality"] = wide["rurality"].fillna("Unknown").astype(str)
    wide["gender"] = wide["gender"].replace("__MISSING__", "Unknown").fillna("Unknown")
    wide["race_mod"] = wide["race_mod"].replace("__MISSING__", "UNKNOWN").fillna("UNKNOWN")
    return wide

def long_to_wide(
    long: pd.DataFrame,
    *,
    duplicate_counts: str = "error",
) -> tuple[pd.DataFrame, list[str]]:
    """Preserve every roster patient; zero-fill only unrecorded conditions.

    Duplicate patient-condition counts require an explicit aggregation rule.
    Missing counts for a recorded condition remain missing, not zero.
    """
    tables = [long.reindex(columns=[ID] + PERSON_FIELDS)]
    info = pd.concat(tables, ignore_index=True).drop_duplicates()
    info = info.loc[~info.index.duplicated(keep='first')]
    info['count'] = info.groupby(['PatientICN'])['PatientICN'].transform('count')
    info['unknown_count'] = np.where(info['race_mod']=='UNKNOWN', 1,0)
    info['count_unknown'] = info.groupby(['PatientICN'])['unknown_count'].transform('sum')
    info['remove'] = np.where((info['count']>=2) & (info['count_unknown']<info['count']) & (info['race_mod']=='UNKNOWN'), 1, 0)
    info = info.loc[(info['remove']==0)]
    info = info.drop_duplicates(subset=[ID],keep="first")
    group = info.groupby(ID, sort=False)
    conflicts = group[PERSON_FIELDS].nunique(dropna=True).gt(1).sum()
    
    if conflicts.any():
        details = {k: int(v) for k, v in conflicts.items() if v}
        raise ValueError(
            f"Conflicting patient-level values (counts by field): {details}. "
            "Resolve these at a common index date; do not arbitrarily take first."
        )
    # first() selects the first nonmissing value only after consistency checks.
    people = group[PERSON_FIELDS].first()
    people.index.name = ID
    for col in ["gender", "race_mod"]:
        people[col] = people[col].fillna("__MISSING__").astype(str)

    dx = long.loc[long["condition"].notna(), [ID, "condition", "times_diagnosed"]].copy()
    known_counts = dx["times_diagnosed"].dropna()
    if known_counts.lt(1).any() or not np.equal(known_counts, np.floor(known_counts)).all():
        raise ValueError("A recorded condition needs a positive integer count or missing.")
    keys = [ID, "condition"]
    if dx.duplicated(keys).any():
        if duplicate_counts == "error":
            raise ValueError(
                "Repeated patient-condition pairs. Use --duplicate-counts max "
                "for repeated cumulative totals or sum for disjoint counts."
            )
        if duplicate_counts == "sum" and dx["times_diagnosed"].isna().any():
            raise ValueError("Cannot sum counts with missing components.")
        dx = dx.groupby(keys, as_index=False)["times_diagnosed"].agg(duplicate_counts)

    conditions = sorted(dx["condition"].unique().tolist())
    if conditions:
        presence = (
            dx.assign(present=1)
            .pivot(index=ID, columns="condition", values="present")
            .reindex(index=people.index, columns=conditions)
            .fillna(0)
            .astype("int8")
        )
        counts = (
            dx.pivot(index=ID, columns="condition", values="times_diagnosed")
            .reindex(index=people.index, columns=conditions)
            .where(presence.eq(1), 0.0)
        )
        people = people.join(presence.add_prefix("dx__")).join(counts.add_prefix("count__"))
    return people.reset_index(), conditions

CPU times: total: 0 ns
Wall time: 17.9 μs


## 4. Model and interpretation helpers

Age, gender, race and rurality are categorical. Conditions remain simultaneous
presence indicators, with optional transformed diagnosis counts.

All category levels are retained for ridge fitting. The reporting reference is a
contrast, calculated as exp(beta_category - beta_reference). This avoids changing
the ridge penalty merely to display another reference. The direct comparison
tables explicitly identify <35 and RUCC 6 as their references.

Ridge tuning and learned preprocessing remain inside nested cross-validation.
These are conditional associations with current use, not causal treatment effects.

In [11]:
%%time
def make_pipeline(conditions: list[str], use_counts: bool) -> Pipeline:
    parts = [
        ("age_group", OneHotEncoder(handle_unknown="ignore"), ["age_bucket"]),
        ("demographics", OneHotEncoder(handle_unknown="ignore"), ["gender", "race_mod", "rurality"]),
    ]
    if conditions:
        parts.append(("diagnosis_presence", "passthrough", [f"dx__{c}" for c in conditions]))
        if use_counts:
            counts = Pipeline([
                ("log1p", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
                ("impute", SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True)),
                ("scale", StandardScaler(with_mean=False)),
            ])
            parts.append(("diagnosis_counts", counts, [f"count__{c}" for c in conditions]))
    return Pipeline([
        ("prepare", ColumnTransformer(parts, remainder="drop")),
        ("logit", LogisticRegression(solver="lbfgs", max_iter=3000, class_weight=None)),
    ])


def fit_search(pipeline, X, y, seed, jobs):
    folds = min(3, int(y.value_counts().min()))
    if folds < 2:
        raise ValueError("Not enough users/nonusers for inner cross-validation.")
    search = GridSearchCV(
        clone(pipeline), {"logit__C": [0.01, 0.1, 1.0, 10.0]},
        scoring="neg_log_loss",
        cv=StratifiedKFold(folds, shuffle=True, random_state=seed),
        n_jobs=jobs, error_score="raise", refit=True,
    )
    search.fit(X, y)
    return search


def make_comparison_table(model, wide, y):
    names = model.named_steps["prepare"].get_feature_names_out()
    coefficients = dict(zip(names, model.named_steps["logit"].coef_[0]))
    groups = [
        ("Age", "age_bucket", "age_group__age_bucket_", AGE_REFERENCE, AGE_LABELS + ["Unknown"]),
        ("RUCC", "rurality", "demographics__rurality_", RURALITY_REFERENCE,
         [str(i) for i in range(1, 10)] + ["Unknown"]),
        ("Gender", "gender", "demographics__gender_", GENDER_REFERENCE, sorted(wide["gender"].unique())),
        ("Race/ethnicity", "race_mod", "demographics__race_", RACE_REFERENCE, sorted(wide["race_mod"].unique())),
    ]
    records = []
    for group, column, prefix, reference, categories in groups:
        ref_key = prefix + reference
        for category in categories:
            key = prefix + category
            mask = wide[column].eq(category)
            estimated = key in coefficients and ref_key in coefficients
            delta = float(coefficients[key] - coefficients[ref_key]) if estimated else np.nan
            odds_ratio = float(np.exp(delta)) if estimated else np.nan
            records.append({
                "comparison_group": group, "category": category, "reference": reference,
                "comparison_log_odds": delta, "adjusted_odds_ratio": odds_ratio,
                "odds_difference_percent": 100 * (odds_ratio - 1),
                "patients": int(mask.sum()), "observed_users": int(y.loc[mask].sum()),
                "status": ("Reference" if category == reference else "Estimated") if estimated
                          else "Not estimated: category or reference absent",
            })
    return pd.DataFrame(records)

def probability_metrics(y: pd.Series, p: np.ndarray) -> dict:
    return {
        "patients": int(len(y)),
        "observed_users": int(y.sum()),
        "sum_predicted_probabilities": float(p.sum()),
        "observed_use_rate": float(y.mean()),
        "mean_predicted_probability": float(p.mean()),
        "roc_auc": float(roc_auc_score(y, p)),
        "average_precision": float(average_precision_score(y, p)),
        "brier_score": float(brier_score_loss(y, p)),
        "log_loss": float(log_loss(y, p, labels=[0, 1])),
    }

def score_wide(wide: pd.DataFrame, conditions: list[str]) -> pd.DataFrame:
    # Unseen diagnosis columns do not enter a model that never learned them.
    for c in conditions:
        for prefix in ["dx__", "count__"]:
            col = prefix + c
            if col not in wide:
                wide[col] = 0
    return wide

CPU times: total: 0 ns
Wall time: 16.7 μs


## 5. Load data or run the synthetic example

The synthetic example includes all age buckets, all nine RUCC codes, some missing
rurality values, multiple conditions and a rare condition with no observed users.
Its age pattern is generated for demonstration only and is not patient evidence.

In [8]:
df

,PatientICN,age,condition,evidence_group,times_diagnosed,VISN,current_acupuncture_user,gender,race_mod,FIPSCode,rurality
0,1006254996,74.0,Obstructive sleep apnea,Group A - Low/Very Low,1,7,0,Male,WHITE,01031,4
1,1002899088,80.0,Pain syndrome,Group D,3,16,0,Male,WHITE,48315,8
2,1006023186,72.0,Gastroesophageal reflux disease,Group D,3,19,0,Male,HISPANIC OR LATINO,08031,1
3,1019161907,42.0,Cervical spinal fusion,Group D,1,10,0,Male,WHITE,18009,6
4,1061041095,27.0,Low back pain,Group A - Low/Very Low,2,8,0,Male,WHITE,12095,1
...,...,...,...,...,...,...,...,...,...,...,...
17562581,1019311623,40.0,Obstructive sleep apnea,Group A - Low/Very Low,1,20,0,Male,UNKNOWN,53061,1
17562582,1005434061,63.0,Pain syndrome,Group D,1,8,0,Male,HISPANIC OR LATINO,72021,0
17562583,1023477636,37.0,Low back pain,Group A - Low/Very Low,1,6,0,Male,WHITE,51760,1
17562584,1018551959,37.0,Shoulder Pain,Group A - High/Strong,1,15,0,Male,HISPANIC OR LATINO,29095,1


In [27]:
%%time
wide, conditions = long_to_wide(df, duplicate_counts=DUPLICATE_COUNTS)
wide = add_age_buckets(wide)
if wide[OUTCOME].isna().any():
    raise ValueError("Every training patient needs a known 0/1 outcome.")
y = wide[OUTCOME].astype(int)
if y.nunique() != 2 or y.value_counts().min() < 6:
    raise ValueError("Need both users and nonusers, at least six of each for nested CV.")
if OUTER_FOLDS < 2:
    raise ValueError("OUTER_FOLDS must be at least 2.")
# Six is an execution minimum, not assurance of adequate statistical sample size.
feature_columns = ["age_bucket", "gender", "race_mod", "rurality"] + [f"dx__{c}" for c in conditions]
if USE_COUNTS:
    feature_columns += [f"count__{c}" for c in conditions]
X = wide[feature_columns]
assert ID not in X.columns and OUTCOME not in X.columns
print(f"Long rows: {len(df):,}; patients: {len(wide):,}; conditions: {len(conditions)}")
print(f"Observed users: {int(y.sum()):,}; nonusers: {int(y.eq(0).sum()):,}")
display(wide.drop(columns=ID).head())
print("Age buckets:\n", wide["age_bucket"].value_counts().reindex(AGE_LABELS + ["Unknown"], fill_value=0))

Long rows: 17,562,584; patients: 5,231,037; conditions: 98
Observed users: 206,968; nonusers: 5,024,069


,age,gender,race_mod,rurality,current_acupuncture_user,dx__Abdominal pain,dx__Acute pancreatitis,dx__Adjustment and other stress-related disorders,dx__Alcohol use disorder,dx__Allergic rhinitis,...,count__Stroke,count__Temporomandibular joint dysfunction,count__Tension Headache,count__Thoracic disc degeneration,count__Tinnitus,count__Tobacco use disorder,count__Tremor,count__Unspecified mood disorder,count__Urinary tract infection,age_bucket
0,74.0,Male,WHITE,4,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,65-74
1,80.0,Male,WHITE,8,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,75+
2,72.0,Male,HISPANIC OR LATINO,1,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,65-74
3,42.0,Male,WHITE,6,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,35-44
4,27.0,Male,WHITE,1,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,<35


Age buckets:
 age_bucket
<35         439293
35-44       709848
45-54       707492
55-64       948738
65-74       990723
75+        1434943
Unknown          0
Name: count, dtype: int64
CPU times: total: 4min 28s
Wall time: 4min 30s


In [31]:
wide.head(n=10).to_csv('test.csv',index=False)

## 6. Fit and validate

Each patient's validation probability is generated by a model that did not train
on that patient's outcome. Inner cross-validation selects the ridge penalty.
Random folds do not replace later-period or facility-level validation.

Your 0.5% acupuncture-claim condition filter is outcome-derived. These results
validate the model conditional on that supplied list; they do not independently
validate condition selection. Freeze the list from an independent earlier period
or repeat selection within training folds using the original unfiltered data.

In [28]:
%%time
pipeline = make_pipeline(conditions, USE_COUNTS)
outer_folds = min(OUTER_FOLDS, int(y.value_counts().min()))
outer = StratifiedKFold(outer_folds, shuffle=True, random_state=RANDOM_SEED)
oof = np.full(len(wide), np.nan)
fold_number = np.zeros(len(wide), dtype=int)
selected_penalties = []
for fold, (train, test) in enumerate(outer.split(X, y), start=1):
    search = fit_search(pipeline, X.iloc[train], y.iloc[train], RANDOM_SEED + fold, JOBS)
    oof[test] = search.predict_proba(X.iloc[test])[:, 1]
    fold_number[test] = fold
    selected_penalties.append(float(search.best_params_["logit__C"]))
    print(f"Validation fold {fold}/{outer_folds} complete.", flush=True)
assert np.isfinite(oof).all()
final = fit_search(pipeline, X, y, RANDOM_SEED, JOBS)
model = final.best_estimator_
metrics = probability_metrics(y, oof)
print("Final ridge C:", final.best_params_["logit__C"])

CPU times: total: 6min 17s
Wall time: 2min 1s



KeyboardInterrupt


KeyboardInterrupt



## 7. Review performance and calibration

ROC AUC and average precision describe discrimination. Brier score and log loss
assess probability prediction; calibration compares predicted and observed rates.
Check the calibration curve and the overall observed-versus-predicted rate before
using probability sums for planning. Lower Brier score alone does not prove good
calibration. Assess demographic and clinical subgroups and independent data too.

In [ ]:
display(pd.DataFrame([metrics]).T.rename(columns={0: "value"}))
measured, predicted = calibration_curve(y, oof, n_bins=10, strategy="quantile")
calibration = pd.DataFrame({
    "mean_predicted_probability": predicted,
    "observed_fraction_users": measured,
})
display(calibration)
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot([0, 1], [0, 1], "--", color="gray", label="Perfect calibration")
ax.plot(predicted, measured, "o-", label="Out-of-fold model")
ax.set(xlabel="Predicted probability", ylabel="Observed fraction using acupuncture",
       xlim=(0, 1), ylim=(0, 1), title="Calibration" + (" — synthetic demo" if USE_SYNTHETIC_DEMO else ""))
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "calibration.png", dpi=150)
plt.show()

## 8. Interpret patient probabilities and user counts

Sum probabilities rather than counting probabilities above 0.5. In the same cohort,
this sum checks whether the model reproduces observed utilization; it will often be
close to the observed user count. It does not reveal a maximum demand ceiling.

The probability sum among known current nonusers is a propensity summary, **not a
forecast of new users**. To forecast initiation, train on baseline nonusers with a
later new-use outcome and appropriate follow-up. No uncertainty interval or
improved-access intervention is estimated here.

In [ ]:
predictions = wide[[ID, OUTCOME]].copy()
predictions["validation_fold"] = fold_number
predictions["probability_oof"] = oof
count_table = pd.DataFrame([
    {"quantity": "Observed current users", "value": int(y.sum())},
    {"quantity": "Sum of validated probabilities in this cohort", "value": float(oof.sum())},
    {"quantity": "Current nonusers", "value": int(y.eq(0).sum())},
    {"quantity": "Nonuser propensity sum — NOT a new-user forecast", "value": float(oof[y.eq(0)].sum())},
])
display(count_table)
display(predictions.drop(columns=ID).head())

## 9. Conditions, raw coefficients and adjusted comparisons

The age comparison table gives each bucket's adjusted odds relative to <35.
The rurality table compares each RUCC category with code 6.
Unknown values have a separate row when observed; absent categories have no
estimated OR. Patient and user counts accompany each comparison.

An OR of 1.5 means 50% higher odds, not 50 percentage points higher probability.
Condition coefficients compare a recorded diagnosis with no recorded diagnosis of
that same condition, adjusting for the other predictors. The comparison patient
can have other diagnoses. With optional count predictors, the presence coefficient
is additionally conditional on the transformed counts.

No p-values or confidence intervals are estimated. These results do not identify
which patients benefit most from treatment or establish a five-year demand ceiling.

In [ ]:
condition_summary = pd.DataFrame([
    {"condition": c,
     "patients_with_condition": int(wide[f"dx__{c}"].sum()),
     "users_with_condition": int(y[wide[f"dx__{c}"].eq(1)].sum())}
    for c in conditions
], columns=["condition", "patients_with_condition", "users_with_condition"])
coefficients = pd.DataFrame({
    "feature": model.named_steps["prepare"].get_feature_names_out(),
    "ridge_log_odds_coefficient": model.named_steps["logit"].coef_[0],
})
display(condition_summary)
display(coefficients.head(25))
comparison_table = make_comparison_table(model, wide, y)
age_comparisons = comparison_table.loc[comparison_table["comparison_group"].eq("Age")].copy()
rurality_comparisons = comparison_table.loc[comparison_table["comparison_group"].eq("RUCC")].copy()
age_observed = pd.DataFrame({"age_bucket": wide["age_bucket"], "observed": y, "predicted": oof})
age_calibration = age_observed.groupby("age_bucket", observed=True).agg(
    patients=("observed", "size"), observed_users=("observed", "sum"),
    observed_use_rate=("observed", "mean"), mean_predicted_probability=("predicted", "mean"),
).reindex(AGE_LABELS + ["Unknown"]).dropna(subset=["patients"]).reset_index()
print("Adjusted age odds ratios; reference:", AGE_REFERENCE)
display(age_comparisons)
print("Adjusted RUCC odds ratios; reference:", RURALITY_REFERENCE)
display(rurality_comparisons)
display(age_calibration)


## 11. Save outputs

The new age_bucket_odds_ratios.csv directly compares age buckets with <35.
rurality_odds_ratios.csv compares RUCC with code 6.
adjusted_odds_ratios.csv includes all categorical comparisons.
age_bucket_calibration.csv shows observed versus validated predicted use by age.

The usual raw coefficients, patient probabilities, condition counts, fitted model
and summary are also saved in this version's separate output directory.
Changing age encoding requires a complete rerun; results from the earlier spline
model cannot be relabeled as bucket estimates.

In [ ]:
summary = {
    "data_mode": "synthetic_demo" if USE_SYNTHETIC_DEMO else "input_data",
    "interpretation": "Current-utilization propensity; not a maximum or initiation forecast.",
    "validation": "Nested stratified cross-validation; conditional on the supplied condition list.",
    "use_counts": USE_COUNTS, "conditions": len(conditions),
    "age_encoding": "categorical_buckets", "age_buckets": AGE_LABELS,
    "reporting_references": {"age_bucket": AGE_REFERENCE, "rurality": RURALITY_REFERENCE,
                             "gender": GENDER_REFERENCE, "race": RACE_REFERENCE},
    "missing_age": int(wide["age_bucket"].eq("Unknown").sum()),
    "missing_rurality": int(wide["rurality"].eq("Unknown").sum()),
    "out_of_fold_metrics": metrics,
    "current_nonusers": int(y.eq(0).sum()),
    "nonuser_propensity_sum_NOT_new_user_forecast": float(oof[y.eq(0)].sum()),
    "outer_fold_C": selected_penalties,
    "final_C": float(final.best_params_["logit__C"]),
    "limitations": [
        "Condition selection from the same outcomes is not independently validated.",
        "Counts/conditions must precede the outcome and exclude acupuncture-generated records.",
        "Probability sums require representative sampling, comparable access and calibration.",
        "No maximum, initiation forecast, uncertainty interval, or causal expansion scenario is estimated.",
        "Clinical validity, decision impact, patient benefit and resource optimization: Data Insufficient.",
    ],
}
if target_summary is not None:
    summary["target_population"] = target_summary
    target_predictions.to_csv(OUTPUT_DIR / "target_probabilities.csv", index=False)
#wide.to_csv(OUTPUT_DIR / "patients_wide.csv", index=False)
predictions.to_csv(OUTPUT_DIR / "patient_probabilities.csv", index=False)
calibration.to_csv(OUTPUT_DIR / "calibration.csv", index=False)
condition_summary.to_csv(OUTPUT_DIR / "condition_counts.csv", index=False)
coefficients.to_csv(OUTPUT_DIR / "model_coefficients.csv", index=False)
comparison_table.to_csv(OUTPUT_DIR / "adjusted_odds_ratios.csv", index=False)
age_comparisons.to_csv(OUTPUT_DIR / "age_bucket_odds_ratios.csv", index=False)
rurality_comparisons.to_csv(OUTPUT_DIR / "rurality_odds_ratios.csv", index=False)
age_calibration.to_csv(OUTPUT_DIR / "age_bucket_calibration.csv", index=False)
joblib.dump({
    "model": model, "conditions": conditions, "feature_columns": feature_columns,
    "use_counts": USE_COUNTS, "sklearn_version": sklearn.__version__,
    "age_buckets": AGE_LABELS, "age_reference": AGE_REFERENCE,
    "rurality_reference": RURALITY_REFERENCE,
}, OUTPUT_DIR / "model.joblib")
(OUTPUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print("Saved results to", OUTPUT_DIR.resolve())

## Data that would improve the analysis

| Additional data | Necessity (1–5) | Likely source and accessibility |
|---|---:|---|
| Baseline/outcome dates, observation length and full population denominator | 5 | Health-system EHR/claims; nonpublic |
| Whether counts include acupuncture encounters | 5 | Claims service/date linkage; system/payer, nonpublic |
| Independent condition-selection period and original unfiltered conditions | 5 for validating the filter | Original claims/EHR extract; nonpublic |
| Facility, referral/access, wait time and travel distance | 4 for utilization; 5 for expansion | Operational/EHR records; primarily nonpublic |
| Eligibility, contraindications, willingness and service capacity | 5 for achievable demand | Clinical review, surveys and scheduling records; nonpublic |
| Condition severity/duration, comorbidity detail and social needs | 4 | EHR/clinical instruments; nonpublic |

Clinical validity, decision impact, patient-centered benefits and resource
optimization are **Data Insufficient** in this utilization-only model.

To compare presence-only and counts models, change `USE_COUNTS`, restart and rerun.
Selecting a preferred version from those validation results requires independent
confirmation or selection within the inner validation process.

References: [logistic regression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html),
[age categories](https://pandas.pydata.org/docs/reference/api/pandas.cut.html),
[one-hot encoding](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html),
[calibration](https://scikit-learn.org/stable/modules/calibration.html),
[pandas pivot](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.pivot.html),
[prediction-model development](https://www.bmj.com/content/386/bmj-2023-078276).